In [0]:
# ==========================================
# 06. INGESTA BRONZE - RESEÑAS
# Proyecto: ElectroCasa
# Método: Auto Loader (JSON)
# ==========================================

RUTA_RESENAS = "/Volumes/proyecto_final/default/vol_landing/resenas"

ARCHIVO = f"{RUTA_RESENAS}/resenas_lote_01_2026-07-31.json"

df_resenas = (
    spark.read
    .option("multiLine", "true")
    .json(ARCHIVO)
)

print("ESQUEMA DEL ARCHIVO:")
df_resenas.printSchema()

print("TOTAL DE REGISTROS:", df_resenas.count())

display(df_resenas.limit(10))

In [0]:
# ==========================================
# CONFIGURACIÓN AUTO LOADER - RESEÑAS
# ==========================================

from pyspark.sql.functions import current_timestamp, col

CATALOGO = "proyecto_final"

RUTA_ORIGEN = (
    "/Volumes/proyecto_final/default/vol_landing/resenas"
)

RUTA_CHECKPOINT = (
    "/Volumes/proyecto_final/default/vol_landing/"
    "_checkpoints/resenas"
)

TABLA_DESTINO = f"{CATALOGO}.bronze.resenas"

# Conservamos la estructura anidada del JSON
ESQUEMA_RESENAS = df_resenas.schema

print("Origen:", RUTA_ORIGEN)
print("Checkpoint:", RUTA_CHECKPOINT)
print("Destino:", TABLA_DESTINO)

print("ESQUEMA CONFIGURADO:")
df_resenas.printSchema()

In [0]:
# ==========================================
# INGESTA BRONZE - RESEÑAS CON AUTO LOADER
# ==========================================

# Lectura incremental de archivos JSON
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(ESQUEMA_RESENAS)
    .load(RUTA_ORIGEN)
)

# Agregar campos de trazabilidad
df_bronze = (
    df_stream
    .withColumn("archivo_origen", col("_metadata.file_name"))
    .withColumn("fecha_ingesta", current_timestamp())
)

# Escritura incremental en Delta
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", RUTA_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLA_DESTINO)
)

query.awaitTermination()

print("Ingesta Bronze de Reseñas finalizada correctamente.")

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.resenas
GROUP BY archivo_origen
ORDER BY archivo_origen;